**Описание задачи**  
Добро пожаловать в 2912 год, где ваши навыки работы с данными потребуются для решения космической загадки. Мы получили сообщение с расстояния в четыре световых года, и дела обстоят не лучшим образом.

Космический корабль «Астра» — межзвездный пассажирский лайнер, запущенный месяц назад.

С почти 13 000 пассажиров на борту судно отправилось в свой первый рейс, перевозя эмигрантов из нашей Солнечной системы на три новые пригодные для жизни экзопланеты, вращающиеся вокруг ближайших звезд.
Огибая систему Альфа Центавра по пути к первому пункту назначения — знойной планете 55 Cancri E, — космический корабль «Астра» столкнулся с пространственно-временной аномалией, скрытой в пылевом облаке. Хотя корабль остался цел, почти половина пассажиров была перенесена в альтернативное измерение!

Чтобы помочь спасателям и вернуть потерянных пассажиров, вам предстоит предсказать, кого из них перенесла аномалия, используя записи, извлеченные из поврежденной компьютерной системы корабля.

Помогите спасти их и изменить историю!

**Задание**

Ваша задача — предсказать, был ли пассажир перенесен в альтернативное измерение во время столкновения космического корабля «Астра» с пространственно-временной аномалией. Для этого вам предоставлен набор личных записей, извлеченных из поврежденной компьютерной системы корабля.

Вам необходимо:

реализовать алгоритм классификации, который сможет по имеющимся данным определить, перенесен пассажир или нет;
получить предсказания на тестовых данных (файл public_test.csv);
загрузить .csv файл с предсказаниями.

In [1]:

import pandas as pd


In [3]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("public_test.csv") 

In [4]:
# Проверка загрузки данных
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())
print("\nTest columns:", test_df.columns.tolist())

Train shape: (8693, 14)
Test shape: (2138, 13)

Train columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported']

Test columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']


In [5]:
# Проверка на пропуски
print("Пропуски в train:")
print(train_df.isnull().sum())
print("\nПропуски в test:")
print(test_df.isnull().sum())


Пропуски в train:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Пропуски в test:
PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
dtype: int64


In [6]:
# Заполним пропуски в числовых столбцах средним значением
# Определим числовые столбцы (примерно, исходя из описания)
numeric_columns = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in numeric_columns:
    if col in train_df.columns:
        mean_val = train_df[col].mean()
        train_df[col] = train_df[col].fillna(mean_val)
        if col in test_df.columns: # Проверяем, существует ли столбец в тесте
            test_df[col] = test_df[col].fillna(mean_val)

In [7]:
# Заполним пропуски в категориальных и бинарных столбцах наиболее частым значением (mode)
categorical_and_binary_columns = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
for col in categorical_and_binary_columns:
    if col in train_df.columns:
        mode_val = train_df[col].mode()[0] if not train_df[col].mode().empty else 'Unknown' # Обработка случая без моды
        train_df[col] = train_df[col].fillna(mode_val)
        if col in test_df.columns: # Проверяем, существует ли столбец в тесте
            test_df[col] = test_df[col].fillna(mode_val)

/tmp/ipykernel_110697/2379943577.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df[col] = train_df[col].fillna(mode_val)


In [8]:
# Для столбцов Cabin и Name можно заполнить пропуски строкой 'Unknown'
if 'Cabin' in train_df.columns:
    train_df['Cabin'] = train_df['Cabin'].fillna('Unknown')
    test_df['Cabin'] = test_df['Cabin'].fillna('Unknown')

if 'Name' in train_df.columns:
    train_df['Name'] = train_df['Name'].fillna('Unknown')
    test_df['Name'] = test_df['Name'].fillna('Unknown')


In [9]:
# Создаем новую колонку с объединёнными текстовыми признаками для train_df
# Используем указанные вами колонки: HomePlanet, CryoSleep, Cabin, Destination, Name
train_df['combined_text'] = (
    train_df['HomePlanet'].astype(str) +
    ' ' + train_df['CryoSleep'].astype(str) +
    ' ' + train_df['Cabin'].astype(str) +
    ' ' + train_df['Destination'].astype(str) +
    ' ' + train_df['Name'].astype(str)
)

In [10]:
# Для test_df
test_df['combined_text'] = (
    test_df['HomePlanet'].astype(str) +
    ' ' + test_df['CryoSleep'].astype(str) +
    ' ' + test_df['Cabin'].astype(str) +
    ' ' + test_df['Destination'].astype(str) +
    ' ' + test_df['Name'].astype(str)
)


In [11]:
# Проверим результат объединения
print("train_df['combined_text'] (первые 5 строк):")
print(train_df['combined_text'].head())

print("\ntest_df['combined_text'] (первые 5 строки):")
print(test_df['combined_text'].head())

train_df['combined_text'] (первые 5 строк):
0     Europa False B/0/P TRAPPIST-1e Maham Ofracculy
1         Earth False F/0/S TRAPPIST-1e Juanna Vines
2       Europa False A/0/S TRAPPIST-1e Altark Susent
3        Europa False A/0/S TRAPPIST-1e Solam Susent
4    Earth False F/1/S TRAPPIST-1e Willy Santantines
Name: combined_text, dtype: object

test_df['combined_text'] (первые 5 строки):
0         Earth True G/73/S TRAPPIST-1e Nelly Richan
1        Mars True F/1631/S TRAPPIST-1e Bleark Weeke
2     Earth False F/885/P 55 Cancri e Courta Johnsby
3    Europa False A/31/S TRAPPIST-1e Jabbab Entenedy
4     Earth True G/958/P TRAPPIST-1e Blancy Moongton
Name: combined_text, dtype: object


In [12]:
# Подготовим признаки для модели
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Определим числовые признаки (все, кроме текстовых и идентификаторов)
all_columns = set(train_df.columns)
# Исключаем целевую переменную, текстовый признак и идентификатор
exclude_cols = {'Transported', 'combined_text', 'PassengerId', 'Name', 'Cabin'} # Убираем Name и Cabin, так как они уже в combined_text
potential_num_cols = all_columns - exclude_cols
# Фильтруем только действительно числовые колонки
num_cols = []
for col in potential_num_cols:
    if pd.api.types.is_numeric_dtype(train_df[col]):
        num_cols.append(col)

print("Числовые колонки для модели:", num_cols)

# %%
# Трансформеры
text_transformer = TfidfVectorizer(max_features=1000, stop_words=None) # Можно настроить max_features
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('text', text_transformer, 'combined_text'),
    ('numeric', numeric_transformer, num_cols)
])

# %%
# Определяем признаки (X) и целевую переменную (y)
feature_cols = ['combined_text'] + num_cols
X = train_df[feature_cols]
y = train_df['Transported']

# %%
# Разделение данных
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# %%
# Построение и обучение модели
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

# %%
# Предсказания на валидационной выборке
val_preds = model.predict(X_val)
val_proba = model.predict_proba(X_val)[:, 1]

print(classification_report(y_val, val_preds, target_names=['Не перенесен', 'Перенесен']))
print("ROC AUC на валидации:", roc_auc_score(y_val, val_proba))

# %%
# Обработка тестового датасета
X_test = test_df[feature_cols]

# %%
# Предсказания на тестовом датасете
test_predictions = model.predict(X_test)
# Получение вероятностей для метрики ROC AUC, если она требуется
# test_probabilities = model.predict_proba(X_test)[:, 1]

# %%
# Создание submission файла
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': test_predictions.astype(bool) # Убедимся, что формат соответствует ожидаемому (bool)
})

# %%
# Просмотрим первые строки получившегося DataFrame:
print(submission_df.head())

# %%
# Сохранение submission файла
submission_df.to_csv('submission.csv', index=False)


Числовые колонки для модели: ['RoomService', 'VRDeck', 'VIP', 'CryoSleep', 'Spa', 'ShoppingMall', 'FoodCourt', 'Age']
              precision    recall  f1-score   support

Не перенесен       0.76      0.81      0.79       863
   Перенесен       0.80      0.75      0.78       876

    accuracy                           0.78      1739
   macro avg       0.78      0.78      0.78      1739
weighted avg       0.78      0.78      0.78      1739

ROC AUC на валидации: 0.8481153139996931
  PassengerId  Transported
0     0495_01         True
1     8464_02         True
2     4277_01        False
3     2380_01        False
4     5918_01         True
